In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("../")
from ccfm.ccfm import (
    load_cfm_traces, 
    make_3d_fault_mesh, 
    load_nshm_traces, 
    load_nrcan_traces,
    make_tri_mesh,
    write_cfm_tri_meshes,
    write_cfm_trace_geojson,
    convert_cfm_geojson,
)

In [3]:
nshm_cfm = "../input_fault_data/nshm_pnw_traces_elev.geojson"

In [4]:
us_flts = load_nshm_traces(nshm_cfm)

In [5]:
us_flts[0]

{'geometry': {'type': 'LineString',
  'coordinates': [[-120.06191, 42.76406, 1507.7137451171875],
   [-120.08438, 42.73862, 1488.2530517578125],
   [-120.1216, 42.7124, 1445.5928955078125],
   [-120.12397, 42.70174, 1571.49658203125],
   [-120.15611, 42.68267, 1378.08935546875],
   [-120.16816, 42.65955, 1461.199462890625],
   [-120.18021, 42.63643, 1373.901123046875],
   [-120.17564, 42.62492, 1561.12255859375],
   [-120.17398, 42.61008, 1598.2073974609375],
   [-120.18207, 42.59489, 1330.74853515625],
   [-120.17741, 42.56784, 1576.99072265625],
   [-120.18421, 42.56448, 1426.8660888671875],
   [-120.19464, 42.55426, 1571.10595703125],
   [-120.20106, 42.54377, 1605.0308837890625],
   [-120.20819, 42.5396, 1559.083740234375],
   [-120.21151, 42.53721, 1626.159423828125],
   [-120.21577, 42.52998, 1646.15185546875],
   [-120.22937, 42.50841, 1714.202392578125],
   [-120.23765, 42.4741, 1494.5877685546875],
   [-120.24593, 42.43979, 1463.8604736328125]]},
 'properties': {'nshm_id': 250

In [6]:
can_faults = "../input_fault_data/WesternCanada_QuaternaryFaults_elev.geojson"

In [7]:
canada_include_ids = (
    "wc022",
    "wc023",
    "wc024",
)

In [8]:
can_flts = load_nrcan_traces(can_faults, include_ids=canada_include_ids)

In [9]:
can_flts[0]

{'geometry': {'type': 'LineString',
  'coordinates': [[-123.5213563063903, 48.58661386202933, 87.74839782714844],
   [-123.49311446958163, 48.57517715441958, -0.150000005960464],
   [-123.4884857007428, 48.573383923159454, 1.738354682922363],
   [-123.47677168377433, 48.56636579030377, 32.005672454833984],
   [-123.4702962378297, 48.56045308284302, 9.419601440429688],
   [-123.46533830282407, 48.55800072733982, 13.615113258361816],
   [-123.46036100531019, 48.55514986744354, 60.96564865112305],
   [-123.45793519115306, 48.55405225178743, 63.4093132019043],
   [-123.45551891556843, 48.5533113348371, 61.864986419677734],
   [-123.4531345248716, 48.552177210437215, 62.0],
   [-123.45215391344543, 48.55184109743381, 62.0],
   [-123.45084608575932, 48.55135172514873, 62.0],
   [-123.44832579071188, 48.5506198158058, 62.0],
   [-123.4443568523486, 48.54924441823297, 62.0],
   [-123.44071369231304, 48.54783673154042, 64.6525650024414],
   [-123.43805460916792, 48.54716694090377, 79.4053573608

In [10]:
flts = us_flts + can_flts

len(flts)

150

In [11]:
%%time

meshes = []
tri_meshes = []

for i, flt in enumerate(flts):
    if i < len(flts) - 1:
        print(f"working on fault {str(i+1).zfill(3)} / {len(flts)}", end='\r')
    else:
        print(f"working on fault {str(i+1).zfill(3)} / {len(flts)}", end='\n')
    try:
        flt_mesh = make_3d_fault_mesh(flt, pt_distance=2.0)
        tri_mesh = make_tri_mesh(flt_mesh)
        meshes.append(flt_mesh)
        tri_meshes.append(tri_mesh)
    except Exception as e:
        print(e)
        print(flt['properties']['id'])
        print(i)

working on fault 150 / 150
CPU times: user 2.26 s, sys: 9.5 ms, total: 2.27 s
Wall time: 2.26 s


In [12]:
write_cfm_tri_meshes("../crescent_cfm_files/crescent_cfm_crustal_3d.geojson", tri_meshes, flts, minify=True)

In [13]:
for i, fault in enumerate(flts):
    fault['geometry']['coordinates'] = meshes[i][0]

In [14]:
write_cfm_trace_geojson("../crescent_cfm_files/crescent_cfm_crustal_traces.geojson", flts, minify=False)

In [15]:
run add_metadata.py

/Users/itchy/research/cascadia/crescent_cfm/scripts/add_metadata.py:67: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  gdf_merged.loc[mask, geojson_col] = gdf_merged.loc[mask, csv_version]


Successfully updated ../crescent_cfm_files/crescent_cfm_crustal_traces.geojson
Updated columns: region, primary_state/province, secondary_state/province, slip_sense, fault_level, NSHM sliprates (MedWgtd), QfaultsAgeCat, QfaultsAgeCat.1, reference, 3D_model_constraints, lineage, comments


/Users/itchy/research/cascadia/crescent_cfm/scripts/add_metadata.py:67: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  gdf_merged.loc[mask, geojson_col] = gdf_merged.loc[mask, csv_version]


Successfully updated ../crescent_cfm_files/crescent_cfm_crustal_3d.geojson
Updated columns: region, primary_state/province, secondary_state/province, slip_sense, fault_level, NSHM sliprates (MedWgtd), QfaultsAgeCat, QfaultsAgeCat.1, reference, 3D_model_constraints, lineage, comments


In [16]:
convert_cfm_geojson("../crescent_cfm_files/crescent_cfm_crustal_traces.geojson")

In [17]:
convert_cfm_geojson("../crescent_cfm_files/crescent_cfm_crustal_3d.geojson",
                    outfile_types=['geopackage'])